In [1]:
import warnings
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction import FeatureHasher
from category_encoders import TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor
from joblib import dump, load
import os

In [2]:
import os
import warnings
import numpy as np
import pandas as pd
from joblib import dump, load
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction import FeatureHasher
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, r2_score
from category_encoders import TargetEncoder

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

class ChampionHasher(BaseEstimator, TransformerMixin):
    def __init__(self, n_features=500):
        self.n_features = n_features
        self.hasher = FeatureHasher(n_features=n_features, input_type='string')
        self.seen_champions = set()

    def fit(self, X, y=None):
        for _, row in X.iterrows():
            self._process_team(row['blue_picks'])
            self._process_team(row['red_picks'])
        return self

    def _process_team(self, picks):
        for champ in picks:
            self.seen_champions.add(champ.strip().title())

    def transform(self, X):
        def hash_team(picks):
            normalized = [c.strip().title() for c in picks]
            return self.hasher.transform([normalized]).toarray()[0]

        blue_features = X['blue_picks'].apply(hash_team)
        red_features = X['red_picks'].apply(hash_team)
        return np.hstack([np.vstack(blue_features), np.vstack(red_features)])

class TeamEncoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.encoder = TargetEncoder(smoothing=10)

    def fit(self, X, y):
        teams = pd.concat([X['blue_team'], X['red_team']])
        targets = pd.concat([y, y])
        self.encoder.fit(teams.values.reshape(-1, 1), targets)
        return self

    def transform(self, X):
        blue_enc = self.encoder.transform(X['blue_team'].values.reshape(-1, 1))
        red_enc = self.encoder.transform(X['red_team'].values.reshape(-1, 1))
        return np.hstack([blue_enc, red_enc])

class TowerPredictor:
    def __init__(self):
        self.pipeline = None
        self.model_dir = os.path.join("models", "tower")
        os.makedirs(self.model_dir, exist_ok=True)

    def load_data(self, path):
        df = pd.read_csv(
            path,
            usecols=[
                'gameid', 'side', 'champion', 'teamname', 'result', 'date',
                'participantid', 'league', 'towers', 'opp_towers'
            ],
            dtype={
                'gameid': 'string',
                'side': 'string',
                'champion': 'string',
                'teamname': 'string',
                'result': 'int8',
                'date': 'string',
                'participantid': 'string',
                'league': 'string'
            },
            parse_dates=['date'],
            na_values=['', ' '],
            keep_default_na=True
        )

        df[['towers', 'opp_towers']] = df[['towers', 'opp_towers']].apply(pd.to_numeric, errors='coerce')

        matches = []
        for game_id in df['gameid'].unique():
            game_data = df[df['gameid'] == game_id]
            team_rows = game_data[game_data['participantid'].isin(['100', '200'])]
            player_rows = game_data[~game_data['participantid'].isin(['100', '200'])]

            if len(player_rows) != 10 or len(team_rows) != 2:
                continue

            try:
                blue_players = player_rows[player_rows['side'] == 'Blue']
                red_players = player_rows[player_rows['side'] == 'Red']
                blue_picks = blue_players['champion'].tolist()
                red_picks = red_players['champion'].tolist()

                if len(blue_picks) != 5 or len(red_picks) != 5:
                    continue

                blue_team_row = team_rows[team_rows['side'] == 'Blue'].iloc[0]
                red_team_row = team_rows[team_rows['side'] == 'Red'].iloc[0]

                towers_total = blue_team_row['towers'] + red_team_row['towers']
                if pd.isna(towers_total):
                    continue

                matches.append({
                    'blue_picks': blue_picks,
                    'red_picks': red_picks,
                    'blue_team': blue_team_row['teamname'],
                    'red_team': red_team_row['teamname'],
                    'towers_total': int(towers_total),
                    'date': blue_team_row['date']
                })

            except Exception:
                continue

        print(f"Parsed {len(matches)} matches with tower data.")
        return pd.DataFrame(matches)

    def train(self, data_path, save_path=None):
        if save_path is None:
            save_path = os.path.join(self.model_dir, "tower_model.joblib")

        df = self.load_data(data_path)

        if len(df) < 10:
            print(f"Need at least 10 matches, got {len(df)}")
            return

        X = df[['blue_picks', 'red_picks', 'blue_team', 'red_team']]
        y = df['towers_total']

        self.pipeline = Pipeline([
            ('features', ColumnTransformer([
                ('champions', ChampionHasher(n_features=200), ['blue_picks', 'red_picks']),
                ('teams', TeamEncoder(), ['blue_team', 'red_team'])
            ])),
            ('scaler', StandardScaler()),
            ('regressor', SVR(C=1.0, kernel='rbf'))
        ])

        split = int(0.8 * len(df))
        X_train, X_test = X.iloc[:split], X.iloc[split:]
        y_train, y_test = y.iloc[:split], y.iloc[split:]

        self.pipeline.fit(X_train, y_train)

        y_pred_train = self.pipeline.predict(X_train)
        y_pred_test = self.pipeline.predict(X_test)

        mae_train = mean_absolute_error(y_train, y_pred_train)
        mae_test = mean_absolute_error(y_test, y_pred_test)

        r2_train = r2_score(y_train, y_pred_train)
        r2_test = r2_score(y_test, y_pred_test)

        print(f"\nTraining MAE: {mae_train:.2f}")
        print(f"Validation MAE: {mae_test:.2f}")
        print(f"Training R^2: {r2_train:.2%}")
        print(f"Validation R^2: {r2_test:.2%}")

        dump(self.pipeline, save_path)

    def load(self, model_path=None):
        if model_path is None:
            model_path = os.path.join(self.model_dir, "tower_model.joblib")
        self.pipeline = load(model_path)
        return self

    def predict(self, blue_picks, red_picks, blue_team, red_team):
        if len(blue_picks) != 5 or len(red_picks) != 5:
            return {'error': 'Need exactly 5 champions per team'}

        input_df = pd.DataFrame([{
            'blue_picks': [c.strip().title() for c in blue_picks],
            'red_picks': [c.strip().title() for c in red_picks],
            'blue_team': blue_team,
            'red_team': red_team
        }])

        try:
            predicted_towers = self.pipeline.predict(input_df)[0]
        except Exception as e:
            return {'error': str(e)}

        return {
            'predicted_total_towers': round(float(predicted_towers), 2)
        }


In [3]:
predictor = TowerPredictor()
predictor.train('matches.csv')

result = predictor.predict(
    blue_picks=["Darius", "Nidalee", "Jayce", "Ezreal", "Alistar"],
    red_picks=["Ambessa", "Skarner", "Aurora", "Varus", "Rakan"],
    blue_team="BoostGate Esports",
    red_team="BBL Dark Passage"
)

if 'error' in result:
    print(f"Error: {result['error']}")
else:
    print(f"Predicted Total Towers Destroyed: {result['predicted_total_towers']}")

Parsed 3345 matches with tower data.

Training MAE: 1.17
Validation MAE: 1.70
Training R^2: 36.48%
Validation R^2: -0.33%
Predicted Total Towers Destroyed: 13.1
